In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from pathlib import Path

MODEL_DIR = Path("/content/drive/MyDrive/FactGrid Data Discovery/byt5_akkadian_model")

print("Can be found in the model file", MODEL_DIR.exists())

if MODEL_DIR.exists():
    for file in MODEL_DIR.iterdir():
        print(file.name)

Can be found in the model file True
config.json
generation_config.json
model.safetensors
training_args.bin
added_tokens.json
tokenizer_config.json
checkpoint-17202
checkpoint-17000


In [5]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("PyTorch version:", torch.__version__)
print("Transformers version:", transformers.__version__)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(
    str(MODEL_DIR), local_files_only=True
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    str(MODEL_DIR), local_files_only=True
).to(device)

model.eval()
print("Model loaded successfully!")

PyTorch version: 2.11.0+cpu
Transformers version: 5.16.1
Using device: cpu


Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model loaded successfully!


In [6]:
def predict(text):
    inputs = tokenizer(text, return_tensors="pt").to(device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

text = "a-na"
print("Input:", text)
print("Prediction:", repr(predict(text)))

Input: a-na
Prediction: 'ana'


# ByT5 Akkadian Inference

This notebook loads an existing Akkadian ByT5 checkpoint from
Google Drive and runs a single-token prediction without retraining.

## Setup
1. Obtain access to the shared model folder.
2. Add a shortcut to the folder in my Google Drive.
3. Mount Google Drive and update MODEL_DIR to my folder path.
4. Run the cells in order.

## Observed result
Input: `a-na`
Prediction: `ana`

This example verifies that the checkpoint loads and generates output.
It does not measure accuracy. The observed output contains no explicit
POS tag; the expected output format still needs to be confirmed.